In [7]:
import requests

OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
CURRENT_VARS = (
    "weathercode,windspeed_10m,winddirection_10m,windgusts_10m,"
    "precipitation,rain,snowfall,showers,snow_depth,"
    "cloudcover,cloudcover_low,temperature_2m,apparent_temperature,"
    "relativehumidity_2m,visibility,surface_pressure"
)


def get_weather(lat: float, lon: float) -> dict | None:
    try:
        response = requests.get(
            OPEN_METEO_URL,
            params={
                "latitude": lat,
                "longitude": lon,
                "current": CURRENT_VARS,
                "wind_speed_unit": "ms",
                "timezone": "UTC",
            },
            timeout=10,
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print(f"timeout for ({lat}, {lon}) — skipping")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error {e.response.status_code} for ({lat}, {lon}) — skipping")
    except Exception as e:
        print(f"unexpected error for ({lat}, {lon}): {e}")
    return None


get_weather(-80, 80)

{'latitude': -80.0,
 'longitude': 80.0,
 'generationtime_ms': 13.626575469970703,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 3999.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'weathercode': 'wmo code',
  'windspeed_10m': 'm/s',
  'winddirection_10m': '°',
  'windgusts_10m': 'm/s',
  'precipitation': 'mm',
  'rain': 'mm',
  'snowfall': 'cm',
  'showers': 'mm',
  'snow_depth': 'm',
  'cloudcover': '%',
  'cloudcover_low': '%',
  'temperature_2m': '°C',
  'apparent_temperature': '°C',
  'relativehumidity_2m': '%',
  'visibility': 'm',
  'surface_pressure': 'hPa'},
 'current': {'time': '2026-03-23T00:30',
  'interval': 900,
  'weathercode': 1,
  'windspeed_10m': 5.12,
  'winddirection_10m': 141,
  'windgusts_10m': 7.0,
  'precipitation': 0.0,
  'rain': 0.0,
  'snowfall': 0.0,
  'showers': 0.0,
  'snow_depth': 40.0,
  'cloudcover': 32,
  'cloudcover_low': 32,
  'temperature_2m': -74.1,
  'apparent_temperature': -81.0,


# WeatherAlert - descripción de variables

## Identificación

| Variable | Tipo | Descripción |
|---|---|---|
| `alert_id` | string | ID único generado por hash de lat/lon/hora/weathercode |
| `region_name` | string | Nombre del punto de la grilla (ej: `grid_-80_80`) |
| `latitude` | float | Latitud WGS-84 en grados decimales |
| `longitude` | float | Longitud WGS-84 en grados decimales |
| `elevation_m` | float | Altitud del punto sobre el nivel del mar en metros |

## Severidad (calculada)

| Variable | Tipo | Descripción |
|---|---|---|
| `alert_type` | string | Tipo de evento: `thunderstorm`, `heavy_rain`, `fog`, `snow`, etc. |
| `severity` | string | Nivel: `minor`, `moderate`, `severe`, `extreme` |
| `severity_score` | float | Score 0.0–1.0 calculado desde weathercode + viento + precipitación |

## Condición general

| Variable | Tipo | Descripción |
|---|---|---|
| `weathercode` | int | Código WMO — describe la condición general del tiempo |
| `interval_s` | int | Intervalo de actualización del modelo en segundos (generalmente 900 = 15 min) |

## Viento

| Variable | Tipo | Descripción |
|---|---|---|
| `windspeed_ms` | float | Velocidad del viento a 10m de altura en m/s |
| `winddirection_deg` | float | Dirección del viento en grados (0° = norte, 90° = este) |
| `windgusts_ms` | float | Velocidad máxima de ráfagas en m/s — crítico para operaciones aéreas |

## Precipitación

| Variable | Tipo | Descripción |
|---|---|---|
| `precipitation_mm` | float | Precipitación total acumulada en mm |
| `rain_mm` | float | Lluvia acumulada en mm |
| `snowfall_cm` | float | Nieve caída en cm |
| `showers_mm` | float | Precipitación de chaparrones en mm |
| `snow_depth_m` | float | Nieve acumulada en el suelo en metros |

## Nubosidad

| Variable | Tipo | Descripción |
|---|---|---|
| `cloudcover_pct` | float | Cobertura nubosa total en % (0 = despejado, 100 = cubierto) |
| `cloudcover_low_pct` | float | Cobertura de nubes bajas en % — afecta directamente la visibilidad en despegue/aterrizaje |

## Atmósfera

| Variable | Tipo | Descripción |
|---|---|---|
| `temperature_c` | float | Temperatura del aire a 2m de altura en °C |
| `apparent_temperature_c` | float | Temperatura de sensación térmica en °C — considera viento y humedad |
| `humidity_pct` | float | Humedad relativa en % |
| `visibility_m` | float | Visibilidad horizontal en metros |
| `pressure_hpa` | float | Presión atmosférica en hPa (hectopascales) |

## Timestamp

| Variable | Tipo | Descripción |
|---|---|---|
| `snapshot_ts` | int | Timestamp Unix (segundos) del momento en que se capturó el dato |

In [ ]:
def generate_grid():
    points = []

    # Base grid (medium density global coverage)
    for lat in range(-60, 76, 20):
        for lon in range(-180, 181, 20):
            points.append(
                {"name": f"grid_{lat}_{lon}", "lat": float(lat), "lon": float(lon)}
            )

    # High-density regions (areas with more air traffic / activity)
    high_density_regions = [
        (-50, 60, -130, -60),  # Americas
        (30, 70, -10, 40),  # Europe
        (10, 60, 60, 140),  # Asia
    ]

    for lat_min, lat_max, lon_min, lon_max in high_density_regions:
        for lat in range(lat_min, lat_max, 10):
            for lon in range(lon_min, lon_max, 10):
                points.append(
                    {"name": f"dense_{lat}_{lon}", "lat": float(lat), "lon": float(lon)}
                )

    # Polar regions (minimal coverage)
    for lat in [-80, 80]:
        for lon in range(-180, 181, 60):
            points.append(
                {"name": f"polar_{lat}_{lon}", "lat": float(lat), "lon": float(lon)}
            )

    unique_points = list({(p["lat"], p["lon"]): p for p in points}.values())

    return unique_points


GRID = generate_grid()
len(GRID)

257

: 

In [3]:
import json
import hashlib
import dataclasses
from dataclasses import dataclass
from datetime import datetime, timezone
import pandas as pd


@dataclass
class Weather:
    snapshot_id: str
    region_name: str
    latitude: float
    longitude: float
    elevation_m: float
    weathercode: int
    interval_s: int
    windspeed_ms: float
    winddirection_deg: float
    windgusts_ms: float
    precipitation_mm: float
    rain_mm: float
    snowfall_cm: float
    showers_mm: float
    snow_depth_m: float
    cloudcover_pct: float
    cloudcover_low_pct: float
    temperature_c: float
    apparent_temperature_c: float
    humidity_pct: float
    visibility_m: float
    pressure_hpa: float
    snapshot_ts: int


def safe_float(val, default=0.0) -> float:
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


def safe_int(val, default=0) -> int:
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


def dict2series(lat: float, lon: float, region_name: str, data: dict) -> pd.Series:
    current = data["current"]
    now_ts = int(datetime.now(tz=timezone.utc).timestamp())
    return pd.Series(
        {
            "latitude": lat,
            "longitude": lon,
            "region_name": region_name,
            "elevation_m": data.get("elevation"),
            "weathercode": current.get("weathercode"),
            "interval_s": current.get("interval"),
            "windspeed_ms": current.get("windspeed_10m"),
            "winddirection_deg": current.get("winddirection_10m"),
            "windgusts_ms": current.get("windgusts_10m"),
            "precipitation_mm": current.get("precipitation"),
            "rain_mm": current.get("rain"),
            "snowfall_cm": current.get("snowfall"),
            "showers_mm": current.get("showers"),
            "snow_depth_m": current.get("snow_depth"),
            "cloudcover_pct": current.get("cloudcover"),
            "cloudcover_low_pct": current.get("cloudcover_low"),
            "temperature_c": current.get("temperature_2m"),
            "apparent_temperature_c": current.get("apparent_temperature"),
            "humidity_pct": current.get("relativehumidity_2m"),
            "visibility_m": current.get("visibility"),
            "pressure_hpa": current.get("surface_pressure"),
            "snapshot_ts": now_ts,
        }
    )


def make_snapshot_id(lat: float, lon: float, ts: int) -> str:
    raw = f"{lat:.1f}:{lon:.1f}:{ts // 900}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]


def weather_from_serie(serie: pd.Series) -> Weather:
    now_ts = safe_int(serie["snapshot_ts"])
    lat = safe_float(serie["latitude"])
    lon = safe_float(serie["longitude"])

    return Weather(
        snapshot_id=make_snapshot_id(lat, lon, now_ts),
        region_name=str(serie["region_name"]),
        latitude=lat,
        longitude=lon,
        elevation_m=safe_float(serie["elevation_m"]),
        weathercode=safe_int(serie["weathercode"]),
        interval_s=safe_int(serie["interval_s"]),
        windspeed_ms=safe_float(serie["windspeed_ms"]),
        winddirection_deg=safe_float(serie["winddirection_deg"]),
        windgusts_ms=safe_float(serie["windgusts_ms"]),
        precipitation_mm=safe_float(serie["precipitation_mm"]),
        rain_mm=safe_float(serie["rain_mm"]),
        snowfall_cm=safe_float(serie["snowfall_cm"]),
        showers_mm=safe_float(serie["showers_mm"]),
        snow_depth_m=safe_float(serie["snow_depth_m"]),
        cloudcover_pct=safe_float(serie["cloudcover_pct"]),
        cloudcover_low_pct=safe_float(serie["cloudcover_low_pct"]),
        temperature_c=safe_float(serie["temperature_c"]),
        apparent_temperature_c=safe_float(serie["apparent_temperature_c"]),
        humidity_pct=safe_float(serie["humidity_pct"]),
        visibility_m=safe_float(serie["visibility_m"]),
        pressure_hpa=safe_float(serie["pressure_hpa"]),
        snapshot_ts=now_ts,
    )


def parse_weather(
    lat: float, lon: float, region_name: str, data: dict
) -> Weather | None:
    serie = dict2series(lat, lon, region_name, data)
    return weather_from_serie(serie)


def weather_serializer(snapshot: Weather) -> bytes:
    return json.dumps(dataclasses.asdict(snapshot)).encode("utf-8")

In [5]:
from kafka import KafkaProducer

SERVER = "localhost:9092"
TOPIC_NAME = "weather-feeds"
INTERVAL = 60

producer = KafkaProducer(
    bootstrap_servers=[SERVER],
    value_serializer=weather_serializer,
)

In [8]:
import time

while True:
    cycle_count = 0
    skipped = 0
    errors = 0

    for point in GRID:
        data = get_weather(point["lat"], point["lon"])

        if data is None:
            errors += 1
            continue

        wheater = parse_weather(point["lat"], point["lon"], point["name"], data)

        if wheater is None:
            skipped += 1
            continue

        producer.send(TOPIC_NAME, value=wheater)
        cycle_count += 1

    producer.flush()
    print(f"alerts sent: {cycle_count}")
    time.sleep(INTERVAL)

alerts sent: 594


KeyboardInterrupt: 